In [1]:
from appgeopy import *
from my_packages import *

# Check original MLCW and modeled MLCW

**2025-05-06** I want to check the consistent between original MLCW and modeled MLCW

# Plot MLCW total compaction (Vertical profile)

## Plot a single dataframe

## Plot two dataframe overlapping

## Plot two dataframe side by side

In [2]:
import os
from glob import glob

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

# --- DATA PREPARATION ---


def load_and_preprocess(fpath):
    """Load CSV and calculate cumulative compaction."""
    df = pd.read_csv(fpath, index_col=[0], parse_dates=[0])
    df = df.dropna(axis=1, how="all")
    depths = df.columns.astype(float).values
    compaction = df.iloc[:, ::-1].cumsum(axis=1)
    return df, depths, compaction


def _get_date_values(dates):
    """Convert dates to a 0-1 scale for color mapping."""
    t0, t1 = dates[0].value, dates[-1].value
    return (dates.astype(np.int64) - t0) / (t1 - t0)


# --- PLOTTING CORE ---


def plot_side_by_side(station_data_list, main_title="Comparison"):
    """Plots multiple stations in separate side-by-side panels."""
    cm = 1 / 2.54
    fig, axes = plt.subplots(1, 2, figsize=(29.7 * cm, 21 * cm), sharey=True)

    # 1. Calculate Global X and Y limits for consistent scaling
    all_x = np.concatenate(
        [s["comp"].values.ravel() for s in station_data_list]
    )
    x_lo = np.floor(np.nanmin(all_x) / 10) * 10 - 10
    x_hi = np.ceil(np.nanmax(all_x) / 10) * 10 + 10

    # Get depth range from the first station
    all_depths = station_data_list[0]["depths"]
    y_min, y_max = np.min(all_depths), np.max(all_depths)

    for i, (ax, station) in enumerate(zip(axes, station_data_list)):
        df = station["df"]
        depths = station["depths"]
        comp = station["comp"]
        cmap = plt.get_cmap(station["cmap"])
        cvals = _get_date_values(df.index)

        # 2. Plot the lines
        for date, cval in zip(comp.index, cvals):
            y_mm = comp.loc[date].values
            ax.plot(
                -y_mm,
                depths[::-1],
                "o",
                ls=":",
                ms=3,
                color=cmap(cval),
                lw=0.8,
                alpha=0.7,
            )

        # 3. Add Horizontal Depth Lines
        for d in depths:
            ax.axhline(y=d, color="black", ls=(0, (5, 5)), lw=0.5, alpha=0.2)
            if i == 0:  # Depth labels only on the left panel
                ax.text(
                    x=-x_lo + 2,
                    y=d,
                    s=f" {int(d)}m",
                    va="center",
                    fontsize=8,
                    color="grey",
                )

        # 4. EXPLICIT AXIS INVERSION (Large depth at bottom)
        ax.set_ylim(320, y_min - 10)
        ax.set_xlim(-x_hi, -x_lo)

        # Move X axis to top
        ax.xaxis.set_ticks_position("top")
        ax.xaxis.set_label_position("top")

        ax.set_xlabel(
            "Compaction (mm)", fontsize=11, fontweight="bold", labelpad=10
        )
        ax.set_title(station["label"], fontsize=14, fontweight="bold", pad=30)

        # 5. TICKS AND GRID MODIFICATIONS
        # Set Major (50) and Minor (10) intervals for Y-axis
        ax.yaxis.set_major_locator(ticker.MultipleLocator(50))
        ax.yaxis.set_minor_locator(ticker.MultipleLocator(10))

        # Apply logic to X-axis as well (optional, but keeps it tidy)
        ax.xaxis.set_major_locator(ticker.AutoLocator())
        ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())

        # Set tick direction to 'out' and length to 8 for major
        ax.tick_params(
            axis="both", which="major", direction="out", length=8, width=1.2
        )
        # Set minor ticks slightly shorter (4 or 5) to distinguish from major, or 8 if preferred
        ax.tick_params(
            axis="both", which="minor", direction="out", length=4, width=0.8
        )

        # Grid and spines
        ax.grid(True, which="major", color="grey", alpha=0.15, linestyle="--")
        ax.spines["right"].set_visible(False)
        ax.spines["bottom"].set_visible(False)

    # 6. Final Formatting
    axes[0].set_ylabel("Depth (m)", fontsize=12, fontweight="bold")
    fig.suptitle(main_title, fontsize=20, fontweight="bold", y=0.9)

    plt.tight_layout(rect=[0, 0.03, 1, 0.93])

    return fig, axes

### plot original & modeled

### plot original & reconstructed

### plot reconstructed & 5m regular spaced